# RLSF — scoring the three arms on val

---
## 1 — Preconditions

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import yaml

from src.eval.stylometrics_ci import MAIN_CONDITIONS

PY = sys.executable
COMET_PY = '.venv-comet/bin/python'
SPLIT = 'val'

ARMS = {'RL-Metric': 'w3_0.0', 'RLSF-Judge': 'w3_2.0', 'RLSF-Judge-High': 'w3_6.0'}
ARM_CONDS = [f'rlsf_{cell}' for cell in ARMS.values()]
ALL_CONDS = MAIN_CONDITIONS + ARM_CONDS
print(' '.join(ALL_CONDS))

In [ ]:
VAL = [json.loads(x) for x in open(f'data/splits/{SPLIT}.jsonl') if x.strip()]

# The hypothesis records do not carry the adapter, so which checkpoint wrote them is only
# recoverable from the config Phase B froze. Read it here so the provenance is on the page.
for cell, cond in zip(ARMS.values(), ARM_CONDS):
    path = Path('outputs') / f'{cond}_{SPLIT}.jsonl'
    rows = [json.loads(x) for x in open(path) if x.strip()]
    assert len(rows) == len(VAL), f'{path}: {len(rows)} rows, expected {len(VAL)}'
    assert all(a['input'] == b['input'] for a, b in zip(rows, VAL)), f'{path}: misalignment'
    assert not [r for r in rows if r.get('error') or not r['prediction'].strip()]

    eval_cfg = yaml.safe_load(Path(f'configs/rlsf_eval_{cell}.yaml').read_text())
    sel = json.loads(Path(f'results/rlsf_select_{cell}.json').read_text())
    assert eval_cfg['generator']['adapter_path'] == sel['selected_path'], cell
    print(f"{cond:14s} {len(rows)} segments, {sel['selected']} @ {sel['selected_path']}")

# Every paid call below lands on val. The test split stays sealed.
assert not any(Path('outputs').glob('*_test.jsonl')), 'a test-split output exists'

In [ ]:
!git rev-parse --short HEAD
!git status --short outputs results

---
## 2 — Surface overlap and the register proxy

In [ ]:
!{PY} manage.py eval --conditions {' '.join(ALL_CONDS)} --split {SPLIT}

---
## 3 — COMET

In [ ]:
subprocess.run(
    [COMET_PY, 'manage.py', 'comet', '--conditions', *ARM_CONDS, '--split', SPLIT],
    check=True,
)

---
## 4 — The two raters

In [ ]:
N_CALLS = len(ARM_CONDS) * len(VAL)

u = json.loads(Path('results/judge_gpt_val_usage.json').read_text())['cumulative']
rate_b = u['cost_usd'] / u['calls']
# Estimated, not measured: no usage artefact exists for the Phi_A pass (docs/budget.md).
RATE_A_EST = 6.187e-4
AUTHORISED = 5.74

print(f"Phi_B gpt-5.6-terra   {N_CALLS} calls x ${rate_b:.4e} = ${N_CALLS * rate_b:.2f}  "
      f"(batch, 50% discount; rate measured over {u['calls']} calls)")
print(f"Phi_A claude-haiku-4-5 {N_CALLS} calls x ${RATE_A_EST:.4e} = ${N_CALLS * RATE_A_EST:.2f}  "
      f"(synchronous; rate ESTIMATED)")
print(f"\ntotal ~${N_CALLS * (rate_b + RATE_A_EST):.2f} against ${AUTHORISED:.2f} authorised")
assert N_CALLS * (rate_b + RATE_A_EST) <= AUTHORISED, 'over the authorised band; do not submit'

In [ ]:
# The rubric both raters read is the evaluation one, distinct from the reward rubric the arms
# were trained against. Scoring on the training rubric would be circular.
import hashlib

for path in ('configs/judge_eval.yaml', 'configs/judge_eval_gpt.yaml'):
    c = yaml.safe_load(Path(path).read_text())
    assert c['template_file'] == 'prompts/judge_eval.txt', c['template_file']
    print(f"{path:30s} {c['judge']['model']:20s} tag={c.get('tag') or '(none)'}")

frozen = json.loads(Path('prompts/hashes.json').read_text())['templates']
digest = hashlib.sha256(Path('prompts/judge_eval.txt').read_bytes()).hexdigest()
assert digest == frozen['judge_eval.txt']['sha256'], 'the evaluation rubric has drifted'
print(f'\nrubric verified {digest[:16]}')

In [ ]:
# Phi_B. The Batch API is submit-then-poll; the batch id is persisted before the first poll, so
# an interrupted cell resumes the same batch instead of paying for a second one.
!{PY} manage.py judge_batch --conditions {' '.join(ARM_CONDS)} --split {SPLIT} \
    --config configs/judge_eval_gpt.yaml

In [ ]:
# Phi_A, synchronous: judge_batch rejects a non-OpenAI provider.
!{PY} manage.py judge --conditions {' '.join(ARM_CONDS)} --split {SPLIT} \
    --config configs/judge_eval.yaml

In [ ]:
for tag, path in (('Phi_A', 'results/judge_val_usage.json'),
                  ('Phi_B', 'results/judge_gpt_val_usage.json')):
    if not Path(path).exists():
        print(f'{tag}: no usage artefact ({path})')
        continue
    d = json.loads(Path(path).read_text())
    s, c = d['session'], d['cumulative']
    print(f"{tag} {d['model']:20s} this pass {s['calls']:5d} calls ${s['cost_usd']:.4f}  "
          f"| cumulative {c['calls']:6d} calls ${c['cost_usd']:.4f}")

---
## 5 — Judge intervals and cross-rater agreement

In [ ]:
!{PY} manage.py judge_ci --conditions {' '.join(ALL_CONDS)} --split {SPLIT}
!{PY} manage.py judge_ci --conditions {' '.join(ALL_CONDS)} --split {SPLIT} --tag gpt

In [ ]:
!{PY} manage.py judge_agreement --conditions {' '.join(ALL_CONDS)} --split {SPLIT} --tag_b gpt

---
## 6 — Stylometrics

In [ ]:
# --targets-split train puts the target register itself in the table, as the row every
# condition's distance is measured against.
!{PY} manage.py stylometrics --conditions {' '.join(ALL_CONDS)} --split {SPLIT} \
    --targets-split train

In [ ]:
!{PY} manage.py stylometrics_ci --conditions {' '.join(ALL_CONDS)} --split {SPLIT}

---
## 7 — Held-out register distance against judge score

In [ ]:
import numpy as np

from src.eval.stylometrics import (
    HELDOUT_FEATURES,
    REWARD_FEATURES,
    aggregate,
    bootstrap_draws,
    distance_to_centroid,
    feature_vector,
    signed_z,
    subcentroid,
)

centroid = json.loads(Path('results/stylometrics_centroid_split.json').read_text())
held = subcentroid(centroid, HELDOUT_FEATURES)
reward = subcentroid(centroid, REWARD_FEATURES)

phi = {t: json.loads(Path(p).read_text())
       for t, p in (('A', 'results/judge_val.json'), ('B', 'results/judge_gpt_val.json'))}


def read(cond):
    rows = [json.loads(x) for x in open(f'outputs/{cond}_{SPLIT}.jsonl') if x.strip()]
    return [r['prediction'] for r in rows]


REPORT = ['peft', *ARM_CONDS]
table = {}
for cond in REPORT:
    preds = read(cond)
    agg = aggregate(preds)
    matrix = np.asarray([feature_vector(t) for t in preds if t.strip()], dtype=float)
    dists, _ = bootstrap_draws(matrix, held, n_resamples=2000, seed=42)
    table[cond] = {
        'dist_heldout': distance_to_centroid(agg['mean'], held),
        'ci': (float(np.percentile(dists, 2.5)), float(np.percentile(dists, 97.5))),
        'dist_reward': distance_to_centroid(agg['mean'], reward),
        'z': signed_z(agg['mean'], centroid),
        'phi_a': phi['A'].get(cond, {}).get('mean'),
        'phi_b': phi['B'].get(cond, {}).get('mean'),
    }

print(f"{'condition':16s} {'d_heldout':>10s} {'95% CI':>18s} {'d_reward':>9s} "
      f"{'Phi_A':>7s} {'Phi_B':>7s}")
for cond, r in table.items():
    a = f"{r['phi_a']:.3f}" if r['phi_a'] is not None else '  n/a'
    b = f"{r['phi_b']:.3f}" if r['phi_b'] is not None else '  n/a'
    print(f"{cond:16s} {r['dist_heldout']:10.4f} "
          f"[{r['ci'][0]:7.4f},{r['ci'][1]:7.4f}] {r['dist_reward']:9.4f} {a:>7s} {b:>7s}")
print(f"\nheld out: {HELDOUT_FEATURES}\nreward:   {REWARD_FEATURES}")

In [ ]:
# Read down the omega ordering, not across the row. The Goodhart claim needs both directions in
# the same arms: judge score up, held-out distance up (further from the target register).
init = table['peft']
print(f"{'arm':16s} {'d(Phi_A)':>9s} {'d(Phi_B)':>9s} {'d(heldout)':>11s} {'d(reward)':>10s}")
for cond in ARM_CONDS:
    r = table[cond]
    da = r['phi_a'] - init['phi_a'] if None not in (r['phi_a'], init['phi_a']) else float('nan')
    db = r['phi_b'] - init['phi_b'] if None not in (r['phi_b'], init['phi_b']) else float('nan')
    print(f"{cond:16s} {da:+9.3f} {db:+9.3f} "
          f"{r['dist_heldout'] - init['dist_heldout']:+11.4f} "
          f"{r['dist_reward'] - init['dist_reward']:+10.4f}")
print('\nAgainst peft, the frozen initialization every arm started from.')

In [ ]:
# Which held-out feature moved, signed. A distance that grew says only that something did.
names = HELDOUT_FEATURES + REWARD_FEATURES
print(f"{'condition':16s}" + ''.join(f'{n:>14s}' for n in names))
for cond, r in table.items():
    print(f'{cond:16s}' + ''.join(f"{r['z'][n]:+14.3f}" for n in names))
print('\nSigned z against the target-register centroid; 0 is the target, sign gives the direction.')

---
## 8 — Paired bootstrap

In [ ]:
BOOT = ['peft', *ARM_CONDS]
for metric in ('chrf', 'bleu', 'comet'):
    cmd = (f"{PY} manage.py bootstrap --metric {metric} --conditions {' '.join(BOOT)} "
           f"--split {SPLIT} --baseline peft --out")
    !{cmd}

In [ ]:
!{PY} manage.py bootstrap --metric judge --conditions {' '.join(BOOT)} \
    --split {SPLIT} --baseline peft --out

In [ ]:
# The second rater's scores, written beside the first rather than over them.
!{PY} manage.py bootstrap --metric judge --conditions {' '.join(BOOT)} --split {SPLIT} \
    --baseline peft --judge_tag gpt --out results/bootstrap_judge_gpt_{SPLIT}.json

---
## 9 — Spend, for `docs/budget.md`

In [ ]:
# Rule 4: a call that is not priced understates the total. Both raters wrote a usage artefact
# this pass, so both rows are provider-reported.
total = 0.0
for tag, model, path in (('Phi_A', 'claude-haiku-4-5', 'results/judge_val_usage.json'),
                         ('Phi_B', 'gpt-5.6-terra', 'results/judge_gpt_val_usage.json')):
    d = json.loads(Path(path).read_text())
    s = d['session']
    total += s['cost_usd']
    print(f"| 2026-08-?? | {tag}, three RLSF arms on val | `{model}` | {s['calls']:,} | "
          f"${s['cost_usd']:.4f} | `{path}` |")
print(f"\npass total ${total:.4f} against ${AUTHORISED:.2f} authorised")